# Bucks on-device intent model: fine-tune FunctionGemma 270M

Adapted from Google's Apache-2.0 notebook *Fine-tune FunctionGemma 270M for Mobile Actions* (gemma-cookbook).
Run on one NVIDIA L4 (24 GB) or bigger, e.g. a Brev instance with Jupyter. Expected time: about 20–40 minutes in total.

Before you start:
1. On huggingface.co, open `google/functiongemma-270m-it`, accept the Gemma licence, and create a read token.
2. Upload `dataset.jsonl` (from `ml/intent/` in the repo, made by `gen_data.py`) next to this notebook.

Output: `litertlm/bucks-intent_q8_ekv1024.litertlm`, plus the accuracy of the base and fine-tuned model per language.
Shut the instance down when you are done: it bills by the hour.

In [ ]:
import sys
!{sys.executable} -m ensurepip --upgrade
!{sys.executable} -m pip install -q torch transformers==4.57.1 trl==0.25.1 datasets==4.4.1 pandas accelerate
!{sys.executable} -c "import torch; print(torch.__version__, torch.cuda.is_available())"

In [ ]:
import getpass
from huggingface_hub import login, whoami
login(token=getpass.getpass("Paste your Hugging Face read token, then press Enter: "))
print("Logged in as", whoami()["name"])

In [ ]:
import json, re
import pandas as pd
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

gemma_model = "google/functiongemma-270m-it"
tokenizer = AutoTokenizer.from_pretrained(gemma_model)
import os, subprocess
if not os.path.exists("dataset.jsonl"):  # regenerate from phrases.py (dataset.jsonl is too big to upload through the browser)
    subprocess.run([sys.executable, "gen_data.py"], check=True)
dataset = load_dataset("text", data_files="dataset.jsonl", encoding="utf-8")["train"].shuffle(seed=7)

def apply_format(sample):
    row = json.loads(sample["text"])
    full = tokenizer.apply_chat_template(row["messages"], tools=row["tools"], tokenize=False, add_generation_prompt=False)
    prompt = tokenizer.apply_chat_template(row["messages"][:-1], tools=row["tools"], tokenize=False, add_generation_prompt=True)
    return {"prompt": prompt, "completion": full[len(prompt):], "split": row["metadata"], "lang": row["lang"]}

processed = dataset.map(apply_format)
train_dataset = processed.filter(lambda e: e["split"] == "train")
held_out = processed.filter(lambda e: e["split"] == "eval")
val_dataset = held_out.select(range(0, len(held_out), 2))    # picks the best checkpoint
test_dataset = held_out.select(range(1, len(held_out), 2))   # reported score, never used for choosing
eval_dataset = val_dataset
longest = max(len(tokenizer.tokenize(e["prompt"] + e["completion"])) for e in processed)
max_token_count = longest + 100
print(len(train_dataset), "train,", len(eval_dataset), "eval, max_length", max_token_count)
print(processed[0]["prompt"][-300:], "\n---\n", processed[0]["completion"])

## Scoring (same rules as `rules_baseline.py`, so the numbers compare)

In [ ]:
def extract_function_call(model_output):
    results = []
    for raw in re.findall(r"<start_function_call>(.*?)<end_function_call>", model_output, re.DOTALL):
        if not raw.strip().startswith("call:") or "{" not in raw:
            continue
        pre, args_segment = raw.split("{", 1)
        args_content = args_segment.strip()
        if args_content.endswith("}"):
            args_content = args_content[:-1]
        args = {m.group("key").strip(): m.group("value")
                for m in re.finditer(r"(?P<key>[^:,]*?):<escape>(?P<value>.*?)<escape>", args_content, re.DOTALL)}
        results.append({"name": pre.replace("call:", "").strip(), "arguments": args})
    return results

def norm_args(args):
    out = {}
    for k, v in (args or {}).items():
        v = str(v).strip().lower()
        if k == "destination":
            v = v.split(",")[0].split(" ")[0]
        out[k] = re.sub(r"^(a|an|the) ", "", v)
    return out

def evaluate(pipe, ds, show_misses=0):
    rows = []
    for e in ds:
        row = json.loads(e["text"])
        want = row["messages"][2]["tool_calls"][0]["function"]
        out = pipe(e["prompt"], max_new_tokens=64, do_sample=False, skip_special_tokens=False)[0]["generated_text"][len(e["prompt"]):]
        calls = extract_function_call(out)
        got = calls[0] if calls else {"name": "help", "arguments": {}}
        action = got["name"] == want["name"]
        rows.append({"lang": e["lang"], "user": row["messages"][1]["content"], "want": want, "got": got,
                     "action": action, "full": action and norm_args(got["arguments"]) == norm_args(want["arguments"])})
    df = pd.DataFrame(rows)
    print(f"right action {df.action.mean():.0%}, right action and details {df.full.mean():.0%}")
    print(df.groupby("lang")[["action", "full"]].mean().round(2))
    for _, r in df[~df.full].head(show_misses).iterrows():
        print(f"  [{r.lang}] {r.user!r} -> {r.got} (want {r.want})")
    return df

## Fine-tune

In [ ]:
import torch
from trl import SFTConfig, SFTTrainer

output_dir = "bucks-intent-functiongemma"
args = SFTConfig(
    output_dir=output_dir,
    num_train_epochs=3,                 # round 1 overfitted after ~1.5 epochs
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    logging_strategy="steps", logging_steps=50,
    eval_strategy="steps", eval_steps=50,
    save_strategy="steps", save_steps=50, save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    learning_rate=1e-5,
    lr_scheduler_type="cosine",
    max_length=max_token_count,
    gradient_checkpointing=True,
    packing=False,
    optim="adamw_torch_fused",
    bf16=True,
    completion_only_loss=True,
    report_to="none",
)
base_model = AutoModelForCausalLM.from_pretrained(gemma_model, device_map="auto", dtype=torch.bfloat16, attn_implementation="eager")
base_model.config.pad_token_id = tokenizer.pad_token_id
trainer = SFTTrainer(model=base_model, args=args, train_dataset=train_dataset, eval_dataset=eval_dataset)
trainer.train()
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

In [ ]:
tuned = AutoModelForCausalLM.from_pretrained(output_dir, device_map="auto")
tuned_pipe = pipeline("text-generation", model=tuned, tokenizer=AutoTokenizer.from_pretrained(output_dir))
tuned_scored = evaluate(tuned_pipe, test_dataset, show_misses=30)
tuned_scored.to_json("eval_finetuned.json", orient="records", force_ascii=False)

## Save the model for download

Converting to `.litertlm` needs no GPU and failed against torch 2.15 here, so it happens on a normal computer afterwards.

In [ ]:
import zipfile, os; d = output_dir; z = zipfile.ZipFile("bucks-intent-checkpoint.zip", "w", zipfile.ZIP_DEFLATED); [z.write(os.path.join(d, f), f) for f in sorted(os.listdir(d)) if os.path.isfile(os.path.join(d, f))]; z.write("eval_finetuned.json"); z.close(); print(round(os.path.getsize("bucks-intent-checkpoint.zip") / 1e6, 1), "MB")

Download `bucks-intent-checkpoint.zip`, then delete the instance: it bills by the hour.